### 1. Preprocessing
- 기존 EMI 방법을 위한 preprocessing을 쓰지 않고, 기본 text를 그대로 가져오되 citations, URL만 제거한 텍스트로 BERT model에 돌리기로 결정
- replication_materials 참고해서 제거하기

In [ ]:
pip install beautifulsoup4 contractions emoji

In [24]:
import pandas as pd
df = pd.read_feather('/Users/juheechoi/Desktop/cmv_mft/dataset/final_dataset_mf_w2v_lenAdj.feather')
df.head()

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,cheating_post_new,ingroup_post_new,outgroup_post_new,authority_post_new,subversion_post_new,sanctity_post_new,degradation_post_new,virtue_post_new,vice_post_new,moral_post_new
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,0.980850,0.514964,0.265883,-0.047656,0.946012,0.332582,2.756070,0.501672,2.091995,1.42468
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,0.980850,0.514964,0.265883,-0.047656,0.946012,0.332582,2.756070,0.501672,2.091995,1.42468
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,0.980850,0.514964,0.265883,-0.047656,0.946012,0.332582,2.756070,0.501672,2.091995,1.42468
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,0.980850,0.514964,0.265883,-0.047656,0.946012,0.332582,2.756070,0.501672,2.091995,1.42468
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,0.226709,0.284522,-0.854095,0.626103,-0.479381,-0.637113,-1.265288,0.178187,-0.905306,-0.40728


In [25]:
df.drop(columns=['moral_comment', 'virtue_comment',
       'vice_comment', 'care_comment', 'harm_comment', 'fairness_comment',
       'cheating_comment', 'ingroup_comment', 'outgroup_comment',
       'authority_comment', 'subversion_comment', 'sanctity_comment',
       'degradation_comment', 'moral_post', 'virtue_post', 'vice_post',
       'care_post', 'harm_post', 'fairness_post', 'cheating_post',
       'ingroup_post', 'outgroup_post', 'authority_post', 'subversion_post',
       'sanctity_post', 'degradation_post', 'care_comment_new',
       'harm_comment_new', 'fairness_comment_new', 'cheating_comment_new',
       'ingroup_comment_new', 'outgroup_comment_new', 'authority_comment_new',
       'subversion_comment_new', 'sanctity_comment_new',
       'degradation_comment_new', 'virtue_comment_new', 'vice_comment_new',
       'moral_comment_new', 'care_post_new', 'harm_post_new',
       'fairness_post_new', 'cheating_post_new', 'ingroup_post_new',
       'outgroup_post_new', 'authority_post_new', 'subversion_post_new',
       'sanctity_post_new', 'degradation_post_new', 'virtue_post_new',
       'vice_post_new', 'moral_post_new'], inplace=True)

In [26]:
import nltk
from nltk.corpus import stopwords
import string
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from bs4 import BeautifulSoup # For more robust HTML/Markdown stripping if needed
import contractions # For expanding contractions like "don't" to "do not"
import emoji # For handling emojis
import time

# --- Download NLTK resources (if not already downloaded) ---
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
# --- Constants ---
STOP_WORDS = set(stopwords.words('english'))
# Add custom stopwords if needed, e.g., common Reddit/CMV terms
CUSTOM_STOP_WORDS = {"cmv", "op", "edit", "eta", "tldr", "dr", "tl", "selftext", "thread_body"}
STOP_WORDS.update(CUSTOM_STOP_WORDS)

LEMMATIZER = WordNetLemmatizer()
MIN_TOKENS_AFTER_PREPROCESSING = 5 # Filter out very short texts

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/juheechoi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/juheechoi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juheechoi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/juheechoi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [27]:
df.head(20)

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,selftext,processed_post,post_date,post_score,title,Topic,Title,Politicality,length_post,delta_ratio
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,"proof: overpopulation, religion, wasting resou...",proof overpopulation religion waste resource p...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,"proof: overpopulation, religion, wasting resou...",proof overpopulation religion waste resource p...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,"proof: overpopulation, religion, wasting resou...",proof overpopulation religion waste resource p...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,"proof: overpopulation, religion, wasting resou...",proof overpopulation religion waste resource p...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,Why should I pay higher taxes when I use less ...,pay high tax use less government service peopl...,1358440202,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0
5,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,3,0,COCO,C,67853eb0f5757a629b4e6ce2,Thompson_S_Sweetback,...,Why should I pay higher taxes when I use less ...,pay high tax use less government service peopl...,1358440202,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0
6,67aa45630bc679138fd9f030,GrizzlyGremlin,16rve8,1.0,1,0,C,C,67853eb0f5757a629b4e6cf2,banebot,...,"This is a reluctant belief of mine, so this su...",reluctant belief mine subreddit could perfect ...,1358457067,24,"I believe monogamy is false, please CMV!",129,Monogamy and Polyamory,0,32,0.0
7,67aa45630bc679138fd9f032,GrizzlyGremlin,16rve8,1.0,1,0,C,C,67853eb0f5757a629b4e6cf9,pervycreeper,...,"This is a reluctant belief of mine, so this su...",reluctant belief mine subreddit could perfect ...,1358457067,24,"I believe monogamy is false, please CMV!",129,Monogamy and Polyamory,0,32,0.0
8,67aa45630bc679138fd9f036,GrizzlyGremlin,16rve8,1.0,1,0,C,C,67853eb0f5757a629b4e6d6f,whale_omelette,...,"This is a reluctant belief of mine, so this su...",reluctant belief mine subreddit could perfect ...,1358457067,24,"I believe monogamy is false, please CMV!",129,Monogamy and Polyamory,0,32,0.0
9,67aa45630bc679138fd9f037,cardswsbound,16rzx1,2.0,1,0,CO,C,67853eb0f5757a629b4e6cea,nix0n,...,I am atheist and she is Christian (not sure on...,atheist christian sure denomination probably m...,1358460478,11,I think the love of my life and I are doomed t...,1,Religion,0,12,0.0


In [28]:
df.drop(columns=['processed_body', 'processed_post'], inplace=True, errors='ignore')  # Drop if they exist

In [ ]:
import re

def clean_thread_body_1(row):
    '''
    OP의 post 글에 있는 citation 제거
    1. thread_body에서 '>'로 시작하고 '\n'으로 끝나는 모든 문자열을 찾습니다.
    2. 찾은 문자열이 selftext에 존재하는지 확인합니다.
    3. 존재한다면 thread_body에서 해당 문자열을 제거합니다.
    4. 최종적으로 수정된 thread_body를 반환합니다.
    '''
    thread_body_list = row['thread_body']
    selftext = row['selftext']
    
    cleaned_body =[]
    for comment in thread_body_list:
        # Find all substrings starting with '>' and ending with '\n'
        matches = re.findall(r'(>.*?\n\n)', comment)
        for match in matches:
            # Check if the match exists in selftext
            if isinstance(row['selftext'], str) and re.sub(r'[>\n]', '', match).strip() in selftext:
                # Remove the match from thread_body
                comment = comment.replace(match, '')
        cleaned_body.append(comment)

    return cleaned_body

def clean_thread_body_2(row, variable_name='thread_body_clean_1'):
    '''
    thread 내에서 댓글끼리의 citation 제거
    1. reference 텍스트 만들기: thread_body[:-1]
    2. thread_body에서 '>'로 시작하고 '\n'으로 끝나는 모든 문자열을 찾습니다.
    3. 찾은 문자열이 reference에 존재하는지 확인합니다.
    4. 존재한다면 thread_body에서 해당 문자열을 제거합니다.
    5. 최종적으로 수정된 thread_body를 반환합니다.
    '''
    thread_body_list = row[variable_name]
    
    # Iterate over a copy of the thread_body list to avoid modifying it while iterating
    cleaned_body = []
    for idx, comment in enumerate(thread_body_list):
        
        # Check if the sentence starts with '>' and ends with '\n\n'
        matches = re.findall(r'(>.*?\n\n)', comment)
        reference = thread_body_list[idx-1] if idx > 0 else None
        if reference:
            for match in matches:
                # Check if the match exists in the reference
                if re.sub(r'[>\n]', '', match).strip() in reference:
                    # Remove the match from thread_body
                    comment = comment.replace(match, '')
        cleaned_body.append(comment)

    return cleaned_body

def remove_urls(row, variable_name='thread_body'): # 없애는 대신 <URL>로 바꿈
    thread_body_list = row[variable_name]
    url_pattern = r'(\(?https?://[^\s]+\)?)'
    cleaned_body = []

    for comment in thread_body_list:
        cleaned_body.append(re.sub(url_pattern, '<URL>', comment))

    return cleaned_body

In [31]:
# Convert comment-level rows into one row per thread so that the thread-level cleaning functions work

def build_thread_level_df(df_in: pd.DataFrame) -> pd.DataFrame:
    df_sorted = df_in.copy()

    if 'comment_date' in df_sorted.columns:
        df_sorted['comment_date'] = pd.to_numeric(df_sorted['comment_date'], errors='coerce')
        df_sorted = df_sorted.sort_values(
            ['thread_id', 'comment_date', 'comment_id'],
            kind='mergesort',
            na_position='last'
        )

    rows = []
    for thread_id, g in df_sorted.groupby('thread_id', sort=False):
        selftext_candidates = [
            x for x in g['selftext'].tolist()
            if isinstance(x, str) and x.strip()
        ]
        selftext = selftext_candidates[0] if selftext_candidates else ''

        thread_body = []
        for body in g['body'].tolist():
            if isinstance(body, str):
                thread_body.append(body)
            elif pd.notna(body):
                thread_body.append(str(body))
            else:
                thread_body.append('')

        rows.append({
            'thread_id': thread_id,
            'thread_body': thread_body,
            'selftext': selftext,
            'post_id': g['post_id'].iloc[0],
            'title': g['title'].iloc[0],
            'OP': g['OP'].iloc[0],
            'thread_size': g['thread_size'].iloc[0],
            'comment_ids': g['comment_id'].tolist(),

        })

    return pd.DataFrame(rows)

In [32]:
# Apply thread-level cleaning functions
thread_df = build_thread_level_df(df)

thread_df['thread_body_clean_1'] = thread_df.apply(clean_thread_body_1, axis=1)
thread_df['thread_body_clean_2'] = thread_df.apply(clean_thread_body_2, axis=1)

thread_df[['thread_id', 'thread_body', 'thread_body_clean_1', 'thread_body_clean_2']].head(3)

,thread_id,thread_body,thread_body_clean_1,thread_body_clean_2
0,67aa45630bc679138fd9f024,[Humanity is stupid relative to what? Aren't w...,[Humanity is stupid relative to what? Aren't w...,[Humanity is stupid relative to what? Aren't w...
1,67aa45630bc679138fd9f026,"[It seems like a lot of this would be normal, ...","[It seems like a lot of this would be normal, ...","[It seems like a lot of this would be normal, ..."
2,67aa45630bc679138fd9f027,[Civilization is a precursor to religion... Hu...,[Civilization is a precursor to religion... Hu...,[Civilization is a precursor to religion... Hu...


In [34]:
thread_df.head()

,thread_id,thread_body,selftext,post_id,title,OP,thread_size,comment_ids,thread_body_clean_1,thread_body_clean_2
0,67aa45630bc679138fd9f024,[Humanity is stupid relative to what? Aren't w...,"proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,[67853eb0f5757a629b4e6d67],[Humanity is stupid relative to what? Aren't w...,[Humanity is stupid relative to what? Aren't w...
1,67aa45630bc679138fd9f026,"[It seems like a lot of this would be normal, ...","proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,[67853eb0f5757a629b4e6d9a],"[It seems like a lot of this would be normal, ...","[It seems like a lot of this would be normal, ..."
2,67aa45630bc679138fd9f027,[Civilization is a precursor to religion... Hu...,"proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,[67853eb0f5757a629b4e6f60],[Civilization is a precursor to religion... Hu...,[Civilization is a precursor to religion... Hu...
3,67aa45630bc679138fd9f029,[We are getting better. Slowly. Painfully slow...,"proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,[67853eb0f5757a629b4e7554],[We are getting better. Slowly. Painfully slow...,[We are getting better. Slowly. Painfully slow...
4,67aa45630bc679138fd9f02b,[1. Long term economic disincentives will not ...,Why should I pay higher taxes when I use less ...,16ralh,I think single people should get the most tax ...,ancillarynipple,4.0,"[67853eb0f5757a629b4e6cdf, 67853eb0f5757a629b4...",[1. Long term economic disincentives will not ...,[1. Long term economic disincentives will not ...


In [ ]:
thread_df['processed_body'] = thread_df.apply(remove_urls, axis=1)
thread_df.drop(columns=['thread_body', 'thread_body_clean_1', 'thread_body_clean_2'], inplace=True)
thread_df = thread_df.explode(['processed_body', 'comment_ids'])
thread_df.head()

,thread_id,selftext,post_id,title,OP,thread_size,comment_ids,processed_body
0,67aa45630bc679138fd9f024,"proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,67853eb0f5757a629b4e6d67,Humanity is stupid relative to what? Aren't we...
1,67aa45630bc679138fd9f026,"proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,67853eb0f5757a629b4e6d9a,"It seems like a lot of this would be normal, a..."
2,67aa45630bc679138fd9f027,"proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,67853eb0f5757a629b4e6f60,Civilization is a precursor to religion... Hun...
3,67aa45630bc679138fd9f029,"proof: overpopulation, religion, wasting resou...",16ra7x,"CMV I think humanity is selfish, stupid, and s...",RealHonestJohn,1.0,67853eb0f5757a629b4e7554,We are getting better. Slowly. Painfully slowl...
4,67aa45630bc679138fd9f02b,Why should I pay higher taxes when I use less ...,16ralh,I think single people should get the most tax ...,ancillarynipple,4.0,67853eb0f5757a629b4e6cdf,1. Long term economic disincentives will not b...


In [42]:
from tqdm import tqdm
thread_df.rename(columns={'comment_ids': 'comment_id'}, inplace=True)
df = df.merge(thread_df[['comment_id', 'processed_body']], how='left', on='comment_id')
df.drop(columns=['body'], inplace=True, errors='ignore')  # Drop original body if it exists

In [45]:
del thread_df

In [46]:
# --- Main Preprocessing Function ---
def preprocess_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    # 2. Lowercasing
    #text = text.lower()

    # 4. Markdown Stripping (Basic - for more complex, consider markdown library)
    # Remove bold/italics markers (keeping the content)
    text = re.sub(r'\*([^*]+)\*', r'\1', text) # Italics *text*
    text = re.sub(r'_([^_]+)_', r'\1', text) # Italics _text_
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text) # Bold **text**
    text = re.sub(r'__([^_]+)__', r'\1', text) # Bold __text__
    text = re.sub(r'~~([^~]+)~~', r'\1', text) # Strikethrough
    # Remove list markers (keeping the content)
    text = re.sub(r'^\s*[\*\-\+]\s+', '', text, flags=re.MULTILINE) # Bullet points
    text = re.sub(r'^\s*\d+\.\s+', '', text, flags=re.MULTILINE) # Numbered lists
    # Remove heading markers
    text = re.sub(r'^#+\s*', '', text, flags=re.MULTILINE)
    # Remove spoiler tags >!text!< (keeping the content)
    text = re.sub(r'>!([^!<]+)!<', r'\1', text)


    # 5. User/Subreddit Mention Removal/Replacement
    text = re.sub(r'/u/\w+|u/\w+', '<USER>', text)
    text = re.sub(r'/r/\w+|r/\w+', '<SUBREDDIT>', text)

    # 6. Emoji Removal (or replacement with text description)
    text = emoji.replace_emoji(text, replace='') # Removes emojis

    # 7. Contraction Expansion
    text = contractions.fix(text)

    # Change abbreviations to full forms
    abbreviations = {
        r'\bAITA\b': 'Am I the Asshole',
        r'\bYTA\b': 'You are the Asshole',
        r'\bNTA\b': 'Not the Asshole',
        r'\bIMO\b': 'In my opinion',
        r'\bIMHO\b': 'In my humble opinion'
    }
    for abbr, full in abbreviations.items():
        text = re.sub(abbr, full, text, flags=re.IGNORECASE)

    return text

In [47]:
df['processed_body_final'] = [preprocess_text(x) for x in tqdm(df['processed_body'], desc="Preprocessing body")]
df['processed_post'] = [preprocess_text(x) for x in tqdm(df['selftext'], desc="Preprocessing post")]
df.head()

Preprocessing post: 100%|██████████| 2062969/2062969 [26:18<00:00, 1306.76it/s] 


,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,post_score,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_body_final,processed_post
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Humanity is stupid relative to what? Aren't we...,Humanity is stupid relative to what? Are not w...,"proof: overpopulation, religion, wasting resou..."
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,"It seems like a lot of this would be normal, a...","It seems like a lot of this would be normal, a...","proof: overpopulation, religion, wasting resou..."
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Civilization is a precursor to religion... Hun...,Civilization is a precursor to religion... Hun...,"proof: overpopulation, religion, wasting resou..."
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,We are getting better. Slowly. Painfully slowl...,We are getting better. Slowly. Painfully slowl...,"proof: overpopulation, religion, wasting resou..."
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0,1. Long term economic disincentives will not b...,Long term economic disincentives will not be v...,Why should I pay higher taxes when I use less ...


In [50]:
def remove_urls_single(row, variable_name='thread_body'): # 없애는 대신 <URL>로 바꿈
    #thread_body_list = row[variable_name]
    url_pattern = r'(\(?https?://[^\s]+\)?)'
    text = row[variable_name]
    text = re.sub(url_pattern, '<URL>', text)

    return text

df['processed_post_final'] = df.apply(remove_urls_single, variable_name='processed_post', axis=1)

In [53]:
# Check whether URLs were actually removed and replaced with <URL>
url_pattern = r'https?://\S+'

changed_mask = df['processed_post'] != df['processed_post_final']
changed_df = df.loc[changed_mask, :].copy()
changed_df

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_body_final,processed_post,processed_post_final
66,67aa45630bc679138fd9f0a3,protagornast,177ikg,2.0,1,0,CO,C,67853eb0f5757a629b4e6d6c,Toby-OrNotToby,...,"[MOD POST] New Sidebar, New Posting Category (...",141,Mods and Delta Comments,0,298,0.0,"Love the updates, guys :) One thing I would sa...","Love the updates, guys :) One thing I would sa...",*This is our first Mod post. You can read our ...,*This is our first Mod post. You can read our ...
67,67aa45630bc679138fd9f0a3,protagornast,177ikg,2.0,2,0,CO,O,67853eb0f5757a629b4e6d95,protagornast,...,"[MOD POST] New Sidebar, New Posting Category (...",141,Mods and Delta Comments,0,298,0.0,Thank you sir/maddam.\n\nI've cut the word cou...,Thank you si<SUBREDDIT>.\n\nI have cut the wor...,*This is our first Mod post. You can read our ...,*This is our first Mod post. You can read our ...
68,67aa45630bc679138fd9f0a5,protagornast,177ikg,3.0,1,0,COC,C,67853eb0f5757a629b4e6d85,knowitall1,...,"[MOD POST] New Sidebar, New Posting Category (...",141,Mods and Delta Comments,0,298,0.0,You need to give a brief TL;DR of that sidebar...,You need to give a brief TL;DR of that sidebar...,*This is our first Mod post. You can read our ...,*This is our first Mod post. You can read our ...
69,67aa45630bc679138fd9f0a5,protagornast,177ikg,3.0,2,0,COC,O,67853eb0f5757a629b4e6d93,protagornast,...,"[MOD POST] New Sidebar, New Posting Category (...",141,Mods and Delta Comments,0,298,0.0,"Alright, I've scaled the sidebar down from 28....","Alright, I have scaled the sidebar down from 2...",*This is our first Mod post. You can read our ...,*This is our first Mod post. You can read our ...
70,67aa45630bc679138fd9f0a5,protagornast,177ikg,3.0,3,0,COC,C,67853eb0f5757a629b4e6db7,knowitall1,...,"[MOD POST] New Sidebar, New Posting Category (...",141,Mods and Delta Comments,0,298,0.0,Everything looks much better now. :D \n\nKee...,Everything looks much better now. :D \n\nKee...,*This is our first Mod post. You can read our ...,*This is our first Mod post. You can read our ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2062958,67aa50f50bc679138ffecd02,SomeGuyOnPluto,e6a4dk,5.0,4,1,COOCD,C,6785432bf5757a629baca350,ThisIsDrLeoSpaceman,...,CMV: The Idea that people should be represente...,136,Changing the World,0,351,0.0,Thanks for the delta! Quick note that you need...,Thanks for the delta! Quick note that you need...,"BEFORE CONTINUING, PLEASE ADDRESS THE VIOLENCE...","BEFORE CONTINUING, PLEASE ADDRESS THE VIOLENCE..."
2062959,67aa50f50bc679138ffecd02,SomeGuyOnPluto,e6a4dk,5.0,5,1,COOCD,D,6785432bf5757a629bacab67,DeltaBot,...,CMV: The Idea that people should be represente...,136,Changing the World,0,351,0.0,"The moderators have confirmed, either contextu...","The moderators have confirmed, either contextu...","BEFORE CONTINUING, PLEASE ADDRESS THE VIOLENCE...","BEFORE CONTINUING, PLEASE ADDRESS THE VIOLENCE..."
2062966,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,1,1,COCOD,C,67853f75f5757a629b60ebe6,millol,...,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,I got birth control pills as a 15 yearold with...,I got birth control pills as a 15 yearold with...,"First of all, I myself am a teenage girl and I...","First of all, I myself am a teenage girl and I..."
2062967,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,3,1,COCOD,C,67853f75f5757a629b60ede3,millol,...,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,No offence taken. I am glad to give you some p...,No offence taken. I am glad to give you some p...,"First of all, I myself am a teenage girl and I...","First of all, I myself am a teenage girl and I..."


In [54]:
del changed_df

In [55]:
df.head()

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_body_final,processed_post,processed_post_final
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Humanity is stupid relative to what? Aren't we...,Humanity is stupid relative to what? Are not w...,"proof: overpopulation, religion, wasting resou...","proof: overpopulation, religion, wasting resou..."
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,"It seems like a lot of this would be normal, a...","It seems like a lot of this would be normal, a...","proof: overpopulation, religion, wasting resou...","proof: overpopulation, religion, wasting resou..."
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Civilization is a precursor to religion... Hun...,Civilization is a precursor to religion... Hun...,"proof: overpopulation, religion, wasting resou...","proof: overpopulation, religion, wasting resou..."
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,We are getting better. Slowly. Painfully slowl...,We are getting better. Slowly. Painfully slowl...,"proof: overpopulation, religion, wasting resou...","proof: overpopulation, religion, wasting resou..."
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,I think single people should get the most tax ...,21,Taxes,1,12,0.0,1. Long term economic disincentives will not b...,Long term economic disincentives will not be v...,Why should I pay higher taxes when I use less ...,Why should I pay higher taxes when I use less ...


In [ ]:
df.drop(columns=['processed_body', 'processed_post'], axis=1, inplace=True)
df.rename(columns={'processed_body_final': 'processed_body', 'processed_post_final': 'processed_post'}, inplace=True)
df.head()

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,post_date,post_score,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_post
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Humanity is stupid relative to what? Are not w...,"proof: overpopulation, religion, wasting resou..."
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,"It seems like a lot of this would be normal, a...","proof: overpopulation, religion, wasting resou..."
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Civilization is a precursor to religion... Hun...,"proof: overpopulation, religion, wasting resou..."
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,We are getting better. Slowly. Painfully slowl...,"proof: overpopulation, religion, wasting resou..."
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,1358440202,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0,Long term economic disincentives will not be v...,Why should I pay higher taxes when I use less ...


In [58]:
df.to_feather('/Users/juheechoi/Desktop/cmv_mft/dataset/dataset_wo_url_citation.feather')

## Moderator footnote 제거

`processed_post`의 상당수(2,062,969행 중 204,942행, unique post 기준 13,179 / 95,786개)에 CMV AutoModerator가 붙이는 규칙 안내 footnote가 원문 뒤에 그대로 붙어 있다 (`"Hello, users of CMV! This is a footnote from your moderators. ..."` ~ `"...Happy CMVing!*"`). 이 텍스트는 글쓴이의 실제 발화가 아니므로 moral foundation 추론 전에 제거해야 한다.

- 시작 anchor는 `"This is a footnote from your moderators"`로 잡았다 (footnote가 있는 13,179개 post 전부에서 100% 등장 확인). `"Hello, users of CMV!"` 부분은 일부 post에서 `"CMV"` 단어 유실 등으로 손상돼 있어 anchor로 쓰기엔 불안정했다.
- 끝은 `"Happy CMVing!"`으로 정확히 anchor하지 않고 **문자열 끝까지(`\Z`)** 지운다. footnote는 항상 post의 맨 마지막 문단이고, 6개 post는 원문 자체가 `"Happy CMVing!"`이 나오기 전에 잘려있어(post 길이 제한으로 추정) 그 phrase를 요구하면 놓쳤기 때문이다.
- footnote 앞의 `____` 구분선과 `>` blockquote 마커도 함께 제거한다.
- (참고) footnote가 두 번 연속 붙어있는 post가 17개 있었는데, 위 패턴으로 한 번에 다 제거됨을 확인했다.

In [1]:
import pandas as pd
df = pd.read_feather('/Users/juheechoi/Desktop/cmv_mft/dataset/dataset_wo_url_citation.feather')
df

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,post_date,post_score,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_post
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Humanity is stupid relative to what? Are not w...,"proof: overpopulation, religion, wasting resou..."
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,"It seems like a lot of this would be normal, a...","proof: overpopulation, religion, wasting resou..."
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Civilization is a precursor to religion... Hun...,"proof: overpopulation, religion, wasting resou..."
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,We are getting better. Slowly. Painfully slowl...,"proof: overpopulation, religion, wasting resou..."
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,1358440202,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0,Long term economic disincentives will not be v...,Why should I pay higher taxes when I use less ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2062964,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,3,1,COCOD,C,67854748f5757a629bfdf0a8,Ok_Magician3130,...,1673302582,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,If I have changed you view as to why vegans ar...,Both are very vocal about an inarguably real i...
2062965,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,5,1,COCOD,D,6785474cf5757a629bfe3980,DeltaBot,...,1673302582,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,"The moderators have confirmed, either contextu...",Both are very vocal about an inarguably real i...
2062966,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,1,1,COCOD,C,67853f75f5757a629b60ebe6,millol,...,1405570058,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,I got birth control pills as a 15 yearold with...,"First of all, I myself am a teenage girl and I..."
2062967,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,3,1,COCOD,C,67853f75f5757a629b60ede3,millol,...,1405570058,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,No offence taken. I am glad to give you some p...,"First of all, I myself am a teenage girl and I..."


In [3]:
post = df[['post_id', 'processed_post']].drop_duplicates(subset='post_id')
post

,post_id,processed_post
0,16ra7x,"proof: overpopulation, religion, wasting resou..."
4,16ralh,Why should I pay higher taxes when I use less ...
6,16rve8,"This is a reluctant belief of mine, so this su..."
9,16rzx1,I am atheist and she is Christian (not sure on...
13,16s6sq,"As a fun experiment, I ask that each person wh..."
...,...,...
2059331,ibudhj,[APPL]<URL> [AMZN]<URL> [MSFT]<URL> [NVDA]<UR...
2061227,lrl1zz,What characterizes the American political comm...
2061392,72px7q,This is something I have been thinking about w...
2061585,430j6c,What I think can and should be done with the E...


In [4]:
import re
from tqdm.auto import tqdm

tqdm.pandas(desc="moderator footnote removal")

MODERATOR_FOOTNOTE_PATTERN = re.compile(
    r"(?:\n{1,3}_{3,}\s*)?\n*>?\s*[^\n]{0,60}?This is a footnote from your moderators.*\Z",
    flags=re.IGNORECASE | re.DOTALL,
)


def remove_moderator_footnote(text) -> str:
    if not isinstance(text, str):
        return text
    return MODERATOR_FOOTNOTE_PATTERN.sub("", text).strip()


before_count = post["processed_post"].str.contains(
    "footnote from your moderators", case=False, na=False
).sum()

post["processed_post"] = post["processed_post"].progress_apply(remove_moderator_footnote)

after_count = post["processed_post"].str.contains(
    "footnote from your moderators", case=False, na=False
).sum()

print(f"제거 전 footnote 포함 행: {before_count}")
print(f"제거 후 footnote 포함 행: {after_count}")
post.head()

/Users/juheechoi/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
moderator footnote removal: 100%|██████████| 95786/95786 [01:14<00:00, 1287.17it/s]


제거 전 footnote 포함 행: 13179
제거 후 footnote 포함 행: 0


,post_id,processed_post
0,16ra7x,"proof: overpopulation, religion, wasting resou..."
4,16ralh,Why should I pay higher taxes when I use less ...
6,16rve8,"This is a reluctant belief of mine, so this su..."
9,16rzx1,I am atheist and she is Christian (not sure on...
13,16s6sq,"As a fun experiment, I ask that each person wh..."


In [9]:
post

,post_id,processed_post
0,16ra7x,"proof: overpopulation, religion, wasting resou..."
4,16ralh,Why should I pay higher taxes when I use less ...
6,16rve8,"This is a reluctant belief of mine, so this su..."
9,16rzx1,I am atheist and she is Christian (not sure on...
13,16s6sq,"As a fun experiment, I ask that each person wh..."
...,...,...
2059331,ibudhj,[APPL]<URL> [AMZN]<URL> [MSFT]<URL> [NVDA]<UR...
2061227,lrl1zz,What characterizes the American political comm...
2061392,72px7q,This is something I have been thinking about w...
2061585,430j6c,What I think can and should be done with the E...


In [11]:
df.drop(columns=['processed_post'], inplace=True, errors='ignore')  # Drop if it exists
df = df.merge(post[['post_id', 'processed_post']], how='left', on='post_id')

In [12]:
df

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,post_date,post_score,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_post
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Humanity is stupid relative to what? Are not w...,"proof: overpopulation, religion, wasting resou..."
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,"It seems like a lot of this would be normal, a...","proof: overpopulation, religion, wasting resou..."
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Civilization is a precursor to religion... Hun...,"proof: overpopulation, religion, wasting resou..."
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,We are getting better. Slowly. Painfully slowl...,"proof: overpopulation, religion, wasting resou..."
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,1358440202,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0,Long term economic disincentives will not be v...,Why should I pay higher taxes when I use less ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2062964,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,3,1,COCOD,C,67854748f5757a629bfdf0a8,Ok_Magician3130,...,1673302582,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,If I have changed you view as to why vegans ar...,Both are very vocal about an inarguably real i...
2062965,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,5,1,COCOD,D,6785474cf5757a629bfe3980,DeltaBot,...,1673302582,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,"The moderators have confirmed, either contextu...",Both are very vocal about an inarguably real i...
2062966,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,1,1,COCOD,C,67853f75f5757a629b60ebe6,millol,...,1405570058,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,I got birth control pills as a 15 yearold with...,"First of all, I myself am a teenage girl and I..."
2062967,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,3,1,COCOD,C,67853f75f5757a629b60ede3,millol,...,1405570058,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,No offence taken. I am glad to give you some p...,"First of all, I myself am a teenage girl and I..."


In [13]:
df.to_feather('/Users/juheechoi/Desktop/cmv_mft/dataset/dataset_wo_url_citation.feather')

### 연도별로 dataset 나누기

In [1]:
import pandas as pd
df = pd.read_feather('dataset_wo_url_citation.feather')
df

/Users/choejuhui/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/choejuhui/anaconda3/lib/python3.11/site-packages/pandas/io/feather_format.py:123: FutureWarning: pyarrow.feather.read_feather is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  return feather.read_feather(


,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,post_date,post_score,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_post
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Humanity is stupid relative to what? Are not w...,"proof: overpopulation, religion, wasting resou..."
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,"It seems like a lot of this would be normal, a...","proof: overpopulation, religion, wasting resou..."
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Civilization is a precursor to religion... Hun...,"proof: overpopulation, religion, wasting resou..."
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,1358439851,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,We are getting better. Slowly. Painfully slowl...,"proof: overpopulation, religion, wasting resou..."
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,1358440202,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0,Long term economic disincentives will not be v...,Why should I pay higher taxes when I use less ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2062964,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,3,1,COCOD,C,67854748f5757a629bfdf0a8,Ok_Magician3130,...,1673302582,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,If I have changed you view as to why vegans ar...,Both are very vocal about an inarguably real i...
2062965,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,5,1,COCOD,D,6785474cf5757a629bfe3980,DeltaBot,...,1673302582,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,"The moderators have confirmed, either contextu...",Both are very vocal about an inarguably real i...
2062966,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,1,1,COCOD,C,67853f75f5757a629b60ebe6,millol,...,1405570058,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,I got birth control pills as a 15 yearold with...,"First of all, I myself am a teenage girl and I..."
2062967,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,3,1,COCOD,C,67853f75f5757a629b60ede3,millol,...,1405570058,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,No offence taken. I am glad to give you some p...,"First of all, I myself am a teenage girl and I..."


In [3]:
# Unix timestamp를 연도로 변환
# post_date가 초 단위(timestamp)인 경우 unit='s'를 사용합니다.
df['post_year'] = pd.to_datetime(df['post_date'], unit='s', errors='coerce').dt.year
df

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,post_score,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_post,post_year
0,67aa45630bc679138fd9f024,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d67,uncannylizard,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Humanity is stupid relative to what? Are not w...,"proof: overpopulation, religion, wasting resou...",2013
1,67aa45630bc679138fd9f026,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6d9a,sehablaespanol1104,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,"It seems like a lot of this would be normal, a...","proof: overpopulation, religion, wasting resou...",2013
2,67aa45630bc679138fd9f027,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e6f60,bert_mars,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,Civilization is a precursor to religion... Hun...,"proof: overpopulation, religion, wasting resou...",2013
3,67aa45630bc679138fd9f029,RealHonestJohn,16ra7x,1.0,1,0,C,C,67853eb0f5757a629b4e7554,SteampunkWolf,...,9,"CMV I think humanity is selfish, stupid, and s...",60,Suicide,0,16,0.0,We are getting better. Slowly. Painfully slowl...,"proof: overpopulation, religion, wasting resou...",2013
4,67aa45630bc679138fd9f02b,ancillarynipple,16ralh,4.0,1,0,COCO,C,67853eb0f5757a629b4e6cdf,Thompson_S_Sweetback,...,14,I think single people should get the most tax ...,21,Taxes,1,12,0.0,Long term economic disincentives will not be v...,Why should I pay higher taxes when I use less ...,2013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2062964,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,3,1,COCOD,C,67854748f5757a629bfdf0a8,Ok_Magician3130,...,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,If I have changed you view as to why vegans ar...,Both are very vocal about an inarguably real i...,2023
2062965,67aa59410bc679138f203766,MAXSR388,107rx8p,5.0,5,1,COCOD,D,6785474cf5757a629bfe3980,DeltaBot,...,0,"CMV: Nothing differentiates a ""preachy vegan"" ...",58,Veganism and Vegetarianism,0,87,0.0,"The moderators have confirmed, either contextu...",Both are very vocal about an inarguably real i...,2023
2062966,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,1,1,COCOD,C,67853f75f5757a629b60ebe6,millol,...,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,I got birth control pills as a 15 yearold with...,"First of all, I myself am a teenage girl and I...",2014
2062967,67aa47780bc679138fe15917,Sun_and_Stars,2axbu7,5.0,3,1,COCOD,C,67853f75f5757a629b60ede3,millol,...,3,CMV: Teenage girls should not be given birth c...,68,Adoption and Childbirth,0,180,0.0,No offence taken. I am glad to give you some p...,"First of all, I myself am a teenage girl and I...",2014


In [ ]:
# 연도별로 분리한 후 각각 feather 파일로 저장
output_dir = 'dataset_by_year'
import os
os.makedirs(output_dir, exist_ok=True)

for year, year_df in df.groupby('post_year', sort=True):
    if pd.isna(year):
        continue
    year = int(year)
    output_path = os.path.join(output_dir, f'dataset_{year}.feather')
    year_df.reset_index(drop=True).to_feather(output_path)
    print(f'Saved {len(year_df)} rows to {output_path}')

/Users/choejuhui/anaconda3/lib/python3.11/site-packages/pandas/io/feather_format.py:65: FutureWarning: pyarrow.feather.write_feather is deprecated as of 24.0.0. Use pyarrow.ipc.new_file() / RecordBatchFileWriter instead. Feather V2 is the Arrow IPC file format.
  feather.write_feather(df, handles.handle, **kwargs)


Saved 87684 rows to dataset_by_year/dataset_2013.feather
Saved 115404 rows to dataset_by_year/dataset_2014.feather
Saved 103967 rows to dataset_by_year/dataset_2015.feather
Saved 106060 rows to dataset_by_year/dataset_2016.feather
Saved 150248 rows to dataset_by_year/dataset_2017.feather
Saved 198839 rows to dataset_by_year/dataset_2018.feather
Saved 151978 rows to dataset_by_year/dataset_2019.feather
Saved 187463 rows to dataset_by_year/dataset_2020.feather
Saved 195149 rows to dataset_by_year/dataset_2021.feather
Saved 155642 rows to dataset_by_year/dataset_2022.feather
Saved 251970 rows to dataset_by_year/dataset_2023.feather
Saved 358565 rows to dataset_by_year/dataset_2024.feather


### [0717] Dataset 합치기 (2023, 2024)

In [ ]:
import pandas as pd
df_1 = pd.read_feather('/Users/choejuhui/Desktop/CMV2/outputs/dataset_2024_scored_checkpoint.feather')
df_2 = pd.read_feather('/Users/choejuhui/Desktop/CMV2/outputs/dataset_2024_scores.feather')

/Users/choejuhui/anaconda3/lib/python3.11/site-packages/pandas/io/feather_format.py:123: FutureWarning: pyarrow.feather.read_feather is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  return feather.read_feather(


In [2]:
df_1.head()

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,care_comment,care_post,harm_comment,harm_post,fairness_comment,fairness_post,cheating_comment,cheating_post,loyalty_comment,loyalty_post
0,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,1,0,COCO,C,678548c9f5757a629b16bc37,raidoheadd,...,0.025761,0.06804,0.001573,0.007461,0.000305,0.388647,0.000646,0.983214,0.001065,0.001188
1,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,2,0,COCO,O,678548c9f5757a629b16bc50,Applesauceoutoflove,...,0.014064,0.06804,0.002019,0.007461,0.000325,0.388647,0.000732,0.983214,0.001065,0.001188
2,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,3,0,COCO,C,678548c9f5757a629b16bc62,raidoheadd,...,0.095349,0.06804,0.007577,0.007461,0.000325,0.388647,0.001325,0.983214,0.000677,0.001188
3,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,4,0,COCO,O,678548c9f5757a629b16bc77,Applesauceoutoflove,...,0.015189,0.06804,0.003324,0.007461,0.000278,0.388647,0.000473,0.983214,0.001116,0.001188
4,67aa5c030bc679138f2a5a1c,Applesauceoutoflove,18vkjp3,1.0,1,0,C,C,678548c9f5757a629b16bc52,possibilistic,...,0.119614,0.06804,0.002397,0.007461,0.009559,0.388647,0.699254,0.983214,0.000955,0.001188


In [5]:
df_2.head()

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,betrayal_comment,authority_comment,subversion_comment,purity_comment,degradation_comment,betrayal_post,authority_post,subversion_post,purity_post,degradation_post
0,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,1,0,COCO,C,678548c9f5757a629b16bc37,raidoheadd,...,0.000511,0.000588,0.000817,0.000607,0.040846,0.00094,0.083292,0.00117,0.001226,0.017712
1,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,2,0,COCO,O,678548c9f5757a629b16bc50,Applesauceoutoflove,...,0.000473,0.000688,0.000911,0.000646,0.012432,0.00094,0.083292,0.00117,0.001226,0.017712
2,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,3,0,COCO,C,678548c9f5757a629b16bc62,raidoheadd,...,0.000480,0.000351,0.000744,0.000431,0.010489,0.00094,0.083292,0.00117,0.001226,0.017712
3,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,4,0,COCO,O,678548c9f5757a629b16bc77,Applesauceoutoflove,...,0.000970,0.001082,0.000767,0.001245,0.017442,0.00094,0.083292,0.00117,0.001226,0.017712
4,67aa5c030bc679138f2a5a1c,Applesauceoutoflove,18vkjp3,1.0,1,0,C,C,678548c9f5757a629b16bc52,possibilistic,...,0.000677,0.002673,0.078078,0.000561,0.010170,0.00094,0.083292,0.00117,0.001226,0.017712


In [6]:
df_2.columns

Index(['thread_id', 'OP', 'post_id', 'thread_size', 'comment_depth',
       'is_delta', 'thread_pattern', 'ChallengerOP', 'comment_id', 'author',
       'comment_date', 'comment_score', 'length_comment', 'delta_ratio_c',
       'year', 'selftext', 'post_date', 'post_score', 'title', 'Topic',
       'Title', 'Politicality', 'length_post', 'delta_ratio', 'processed_body',
       'processed_post', 'post_year', 'betrayal_comment', 'authority_comment',
       'subversion_comment', 'purity_comment', 'degradation_comment',
       'betrayal_post', 'authority_post', 'subversion_post', 'purity_post',
       'degradation_post'],
      dtype='object')

In [7]:
df_2 = df_2[['comment_id', 'betrayal_comment', 'authority_comment',
       'subversion_comment', 'purity_comment', 'degradation_comment',
       'betrayal_post', 'authority_post', 'subversion_post', 'purity_post',
       'degradation_post']]

df_1 = df_1.merge(df_2, how='left', on='comment_id')
df_1.head()

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,betrayal_comment,authority_comment,subversion_comment,purity_comment,degradation_comment,betrayal_post,authority_post,subversion_post,purity_post,degradation_post
0,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,1,0,COCO,C,678548c9f5757a629b16bc37,raidoheadd,...,0.000511,0.000588,0.000817,0.000607,0.040846,0.00094,0.083292,0.00117,0.001226,0.017712
1,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,2,0,COCO,O,678548c9f5757a629b16bc50,Applesauceoutoflove,...,0.000473,0.000688,0.000911,0.000646,0.012432,0.00094,0.083292,0.00117,0.001226,0.017712
2,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,3,0,COCO,C,678548c9f5757a629b16bc62,raidoheadd,...,0.000480,0.000351,0.000744,0.000431,0.010489,0.00094,0.083292,0.00117,0.001226,0.017712
3,67aa5c030bc679138f2a5a12,Applesauceoutoflove,18vkjp3,4.0,4,0,COCO,O,678548c9f5757a629b16bc77,Applesauceoutoflove,...,0.000970,0.001082,0.000767,0.001245,0.017442,0.00094,0.083292,0.00117,0.001226,0.017712
4,67aa5c030bc679138f2a5a1c,Applesauceoutoflove,18vkjp3,1.0,1,0,C,C,678548c9f5757a629b16bc52,possibilistic,...,0.000677,0.002673,0.078078,0.000561,0.010170,0.00094,0.083292,0.00117,0.001226,0.017712


In [9]:
column_order = ['thread_id', 'OP', 'post_id', 'thread_size', 'comment_depth',
       'is_delta', 'thread_pattern', 'ChallengerOP', 'comment_id', 'author',
       'comment_date', 'comment_score', 'length_comment', 'delta_ratio_c',
       'year', 'selftext', 'post_date', 'post_score', 'title', 'Topic',
       'Title', 'Politicality', 'length_post', 'delta_ratio', 'processed_body',
       'processed_post', 'post_year', 'care_comment', 'care_post',
       'harm_comment', 'harm_post', 'fairness_comment', 'fairness_post',
       'cheating_comment', 'cheating_post', 'loyalty_comment', 'loyalty_post',
       'betrayal_comment', 'betrayal_post', 'authority_comment', 'authority_post','subversion_comment', 'subversion_post', 'purity_comment', 'purity_post','degradation_comment', 'degradation_post']

df_1 = df_1[column_order]
df_1.to_feather('/Users/choejuhui/Desktop/CMV2/outputs/dataset_2024_scored.feather')


/Users/choejuhui/anaconda3/lib/python3.11/site-packages/pandas/io/feather_format.py:65: FutureWarning: pyarrow.feather.write_feather is deprecated as of 24.0.0. Use pyarrow.ipc.new_file() / RecordBatchFileWriter instead. Feather V2 is the Arrow IPC file format.
  feather.write_feather(df, handles.handle, **kwargs)
